# redback simulations

use redback to simulate tdes, sn at different redshifts

https://redback.readthedocs.io/en/latest/simulation.html#examples

In [ ]:
import numpy as np
import pandas as pd
import os
import bilby
import redback  
from astropy.cosmology import Planck18 as cosmo                                                                                                                   
import matplotlib.pyplot as plt
import matplotlib              
from matplotlib import rcParams
rcParams["font.family"] = "Liberation Serif"
matplotlib.rcParams['text.usetex'] = False 

# FIXME LALsimulation not installed

## Fetch real object photometry

https://redback.readthedocs.io/en/latest/getting_data.html

redback has built in tools to get data from open access, otter catalogs and fink and lasair brokers

In [ ]:
# mag check 

M = -23
z = 1
m = M + cosmo.distmod(z).value
print(f'm = {m:.2f}')

In [ ]:
# which real tdes can we retrieve from open access catalog

oac_path = os.path.join(os.path.dirname(redback.__file__), 'tables', 'OAC_metadata.csv')                                                                          
df = pd.read_csv(oac_path, low_memory=False, on_bad_lines='skip')
tdes = df[df['claimedtype'].str.contains('TDE|tidal', case=False, na=False)]                                                                                      
                                                                                                                                                                
print(f"Total TDEs in redback OAC catalog: {len(tdes)}")                                                                                                          
print(tdes['name'].tolist())

In [ ]:
# search for certain characteristics ie bright                                                                                                                                                
                                                                                                                                                                  
oac_path = os.path.join(os.path.dirname(redback.__file__), 'tables', 'OAC_metadata.csv')                                                                          
df = pd.read_csv(oac_path, low_memory=False, on_bad_lines='skip')
tdes = df[df['claimedtype'].str.contains('TDE|tidal', case=False, na=False)].copy()                                                                               
tdes['maxabsmag'] = pd.to_numeric(tdes['maxabsmag'], errors='coerce')                                                                                             
                                                                                                                                                                
# print brightest                                                                                                                            
brightest = tdes[['name', 'maxabsmag']].dropna().sort_values('maxabsmag').head(20)                                                                                 
print("Brightest TDEs by peak absolute magnitude:")                                                                                                             
print(brightest.to_string(index=False))                                                                                                                           
                                                                                                                                                                
# histogram                                                                                                                                                       
fig, ax = plt.subplots(figsize=(8, 5))                                                                                                                          
tdes['maxabsmag'].dropna().plot.hist(bins=20, ax=ax, edgecolor='black')                                                                                           
ax.set_xlabel('Peak Absolute Magnitude')
ax.set_ylabel('Count')                                                                                                                                            
ax.set_title('Distribution of TDE Peak Absolute Magnitudes (redback OAC catalog)')                                                                              
ax.invert_xaxis()  # brighter = more negative, so flip axis                                                                                                       
plt.tight_layout()                                                                                                                                                
plt.show()  

In [ ]:
# instead of filtering for OAC TDEs, just check brightest objects

oac_path = os.path.join(os.path.dirname(redback.__file__), 'tables', 'OAC_metadata.csv')                                                                          
df = pd.read_csv(oac_path, low_memory=False, on_bad_lines='skip')
df['maxabsmag'] = pd.to_numeric(df['maxabsmag'], errors='coerce')
brightest = df.dropna(subset=['maxabsmag']).sort_values('maxabsmag').head(15)
print(brightest[['name', 'maxabsmag', 'claimedtype', 'redshift']].to_string())                                                                                   

In [ ]:
import otter
import numpy as np

db = otter.Otter()

# query for confirmed TDEs with optical photometry in one step
all_transients = db.query(classification='TDE', unambiguous=True,)# has_uvoir_phot=True)

tdes_with_z = []
for t in all_transients:
    try:
        z = t.get_redshift()
    except (KeyError, TypeError):
        continue
    if z is None:
        continue

    z_val = float(np.median([v for v in (z if isinstance(z, (list, np.ndarray)) else [z]) if v is not None]))
    name = t.get('name', {}).get('default_name', str(t))
    tdes_with_z.append((name, z_val))

tdes_with_z.sort(key=lambda x: x[1], reverse=True)

print(f"Found {len(tdes_with_z)} confirmed TDEs with UVOIR photometry\n")
print(f"{'Rank':<6} {'Name':<25} {'Redshift'}")
print("-" * 45)
for i, (name, z) in enumerate(tdes_with_z[:15], 1):
    print(f"{i:<6} {name:<25} {z:.4f}")

In [ ]:
target_object = '2025aarm'

for t in all_transients:
    name = t.get('name', {}).get('default_name', str(t))
    if target_object in name:
        print(f"Found target object: {name}")
        break

### now load photometry for selected real objects

In [ ]:
# FIXME redback requires to make a directory
# also can try AT2018iih (z=0.212), AT2020ysg, AT2020acka

# load with name
name = 'AT2019cmw'

# redback bug fix
outdir = f'tidal_disruption_event/{name}/uvoir'
os.makedirs(outdir, exist_ok=True)

# either load from otter (tdes only right now)
data = redback.get_data.get_tidal_disruption_event_data_from_otter(transient=name)

# # or load tdes from open access catalog
# data = redback.get_data.get_tidal_disruption_event_data_from_open_transient_catalog_data(transient=name)

# # or load SN from OAC
# data = redback.get_data.get_supernova_data_from_open_transient_catalog_data(transient=name)

# # or query Fink for ZTF objects
# data = redback.get_data.get_fink_data(transient=name, transient_type='supernova')

In [ ]:
# can query FINK and LASAIR also
#FIXME fetch more bands of data.

name = 'ZTF25acfyeke'

data = redback.get_data.get_fink_data(transient=name, transient_type='tidal_disruption_event')

In [ ]:
# create redback transient object from loaded data
exclude_bands = ['W2', 'M2', 'W1', 'U']
bands=data['band'].unique() # utilize all bands available
#bands = [b for b in bands if b not in exclude_bands] # filter out non-optical bands

# # from OTTER TDE (also downloads if not cached)                                                                                                                           
# transient_object = redback.tde.TDE.from_otter(name=name, data_mode="flux_density")  

# # from OAC TDE                                                                                                                                                              
# transient_object = redback.tde.TDE.from_open_access_catalogue(name=name, data_mode="flux_density", active_bands=bands)

# # from OAC Supernova                                                                                                                                                        
# transient_object = redback.supernova.Supernova.from_open_access_catalogue(name=name, data_mode="flux_density", active_bands=bands)                                              
                                                                                                                                                                            
# from Fink — same as OAC since they share the same processed file path                                                                                                   
transient_object = redback.tde.TDE.from_open_access_catalogue(name=name, data_mode="flux_density", active_bands=bands) 

In [ ]:
# plot

# the black points are bands available that havent be set as active                                                                                     
transient_object.plot_data(show=False, title=f"{name} photometry")                                                                                                                                                   
# fig, axes = plt.subplots(1,3, sharex=True, sharey=True, figsize=(12, 5))                                                                                                 
# object.plot_multiband(figure=fig, axes=axes, filters=["g'", "r'", "z'"], show=False)                                                                               

Warning: 

- check if data are in AB or Vega system (redback assumes AB, throws away explicity flagged Vega from processed file)

- check if extinction correction was applied to OAC data (redback assumes no extinction correction applied)

- check filter names are correct. EX: 'r' is treated as SDSS r but may be referring to R band or ZTF/LSST r band which have different filter responses


In [ ]:
# FIXME automate data checks

# check what bands the observations are in
print("Unique bands:", transient_object.unique_bands)   
print("Active bands:", transient_object.active_bands)     

# count photometry points per band
from collections import Counter
all_counts = Counter(transient_object.bands)
print("\nAll bands (including inactive):")
for band, count in sorted(all_counts.items()):
    active = band in transient_object.active_bands
    print(f"  {band}: {count}")

# fit model based on real photometry

In [ ]:
# run inference to extract model best parameters

# tde model options:
# Simple analytic fallback
# Cooling envelope (Metzger 2022)
# Gaussian rise + cooling envelope (Sarin and Metzger 2024)
# Broken power law + cooling envelope (Sarin and Metzger 2024)
# Fitted
# TDEMass
# Mosfit TDE

In [ ]:
sampler = 'dynesty'
model = 'tde_analytical'
nlive=500
outdir = f'redback_inference/{name}/{model}'
os.makedirs(outdir, exist_ok=True)

# # By default all data is used, select bands for speed or if you only trust some of the data/physics.
# bands = ["u", "r"]
# transient_object.active_bands = bands

# start with default priors
priors = redback.priors.get_priors(model=model)
# Fix priors, ie if known through the metadata that was downloaded alongside the data
priors['redshift'] = 0.4030
# tighten temperature prior — TDE blackbodies are typically 10k-50k K at peak
priors['temperature_0'] = bilby.core.prior.Uniform(
    minimum=10000, maximum=60000, name='temperature_0', latex_label='$T_0$ (K)')
# tighten radius prior around physically expected TDE photosphere size
priors['radius_0'] = bilby.core.prior.LogUniform(
    minimum=1e13, maximum=1e16, name='radius_0', latex_label='$R_0$ (cm)')

model_kwargs = dict(frequency=transient_object.filtered_frequencies, output_format='flux_density')

# returns a tde result object
result = redback.fit_model(transient=transient_object, model=model, sampler=sampler,
                           model_kwargs=model_kwargs, prior=priors, sample='rslice', nlive=nlive, npool=4, 
                           resume=True, outdir=outdir, label=f"{name}_nlive{str(nlive)}",
                           plot=False)

# result.transient reconstructs from metadata with wrong path, use transient_object directly
model_func = redback.model_library.all_models_dict[model]

# plot results
result.plot_corner(save=False, show=True)
plt.show()
plt.close('all')
transient_object.plot_lightcurve(
    model=model_func,
    posterior=result.posterior,
    model_kwargs=result.model_kwargs,
    random_models=100,
    save=False,
    show=True,
)
transient_object.plot_multiband_lightcurve(
    model=model_func,
    posterior=result.posterior,
    model_kwargs=result.model_kwargs,
    random_models=100,
    filters=["ztfg", "ztfr"],
    save=False,
    show=True,
)

In [ ]:
# print all the priors for the fit model

priors = redback.priors.get_priors(model='tde_analytical')
priors

In [ ]:
# try cooling envelope model

sampler = 'dynesty'
model = 'cooling_envelope'
nlive=500
outdir = f'redback_inference/{name}/{model}'
os.makedirs(outdir, exist_ok=True)

# # By default all data is used, select bands for speed or if you only trust some of the data/physics.
# bands = ["u", "r"]
# transient_object.active_bands = bands

# start with default priors
priors = redback.priors.get_priors(model=model)
# Fix priors, ie if known through the metadata that was downloaded alongside the data
priors['redshift'] = 0.4030
# free up the physically interesting parameters (all fixed by default)                                                                                                      
priors['mbh_6'] = bilby.core.prior.LogUniform(                                                                                                                              
    minimum=0.1, maximum=100, name='mbh_6', latex_label='$M_{\\rm BH}/10^6 M_\\odot$')                                                                                      
priors['eta'] = bilby.core.prior.Uniform(                                                                                                                                   
    minimum=0.01, maximum=0.4, name='eta', latex_label='$\\eta$')                                                                                                           
priors['alpha'] = bilby.core.prior.Uniform(                                                                                                                                 
    minimum=0.01, maximum=0.4, name='alpha', latex_label='$\\alpha$')                                                                                                       
priors['beta'] = bilby.core.prior.Uniform(                                                                                                                                  
    minimum=1.0, maximum=6.0, name='beta', latex_label='$\\beta$')

model_kwargs = dict(frequency=transient_object.filtered_frequencies, output_format='flux_density')

# returns a tde result object
result = redback.fit_model(transient=transient_object, model=model, sampler=sampler,
                           model_kwargs=model_kwargs, prior=priors, sample='rslice', nlive=nlive, npool=4, 
                           resume=True, outdir=outdir, label=f"{name}_nlive{str(nlive)}",
                           plot=False)

# result.transient reconstructs from metadata with wrong path, use transient_object directly
model_func = redback.model_library.all_models_dict[model]

# plot results
result.plot_corner(save=False, show=True)
plt.show()
plt.close('all')
transient_object.plot_lightcurve(
    model=model_func,
    posterior=result.posterior,
    model_kwargs=result.model_kwargs,
    random_models=100,
    save=False,
    show=True,
)
transient_object.plot_multiband_lightcurve(
    model=model_func,
    posterior=result.posterior,
    model_kwargs=result.model_kwargs,
    random_models=100,
    filters=["ztfg", "ztfr"],
    save=False,
    show=True,
)

In [ ]:
# Experiment with fit: afterglow SED + blackbody via afterglow_and_optical combined model
#
# afterglow_and_optical routes all flat sampled kwargs to both sub-models via **model_kwargs.update(),
# so each sub-model picks up only the parameters it needs — no custom wrapper required.

sampler = 'dynesty'
model = 'afterglow_and_optical'
base_afterglow = 'tophat'
base_optical = 'evolving_blackbody'
nlive = 1000
outdir = f'redback_inference/{name}/{model}_{base_afterglow}'
os.makedirs(outdir, exist_ok=True)

# combined flat priors: tophat afterglow + evolving blackbody
priors = redback.priors.get_priors(model=base_afterglow)
priors.update(redback.priors.get_priors(model=base_optical))

# add host extinction (required by afterglow_and_optical)
priors['av'] = bilby.core.prior.LogUniform(minimum=1e-3, maximum=3.0, name='av', latex_label='$A_V$')

# fix redshift; tighten temperature and radius for TDE-like conditions
priors['redshift'] = 0.4030
priors['temperature_0'] = bilby.core.prior.Uniform(
    minimum=10000, maximum=60000, name='temperature_0', latex_label='$T_0$ (K)')
priors['radius_0'] = bilby.core.prior.LogUniform(
    minimum=1e13, maximum=1e16, name='radius_0', latex_label='$R_0$ (cm)')

# fixed kwargs: sub-model names + frequency + format
# sampled parameters flow through as flat kwargs and get routed to each sub-model automatically
model_kwargs = dict(
    frequency=transient_object.filtered_frequencies,
    output_format='flux_density',
    base_afterglow_model=base_afterglow,
    base_optical_model=base_optical,
    afterglow_kwargs={},
    optical_kwargs={},
)

result = redback.fit_model(transient=transient_object, model=model, sampler=sampler,
                           model_kwargs=model_kwargs, prior=priors, sample='rslice', nlive=nlive, npool=4,
                           resume=True, outdir=outdir, label=f"{name}_nlive{str(nlive)}",
                           plot=False)

model_func = redback.model_library.all_models_dict[model]

result.plot_corner(save=False, show=True)
plt.show()
plt.close('all')

transient_object.plot_lightcurve(
    model=model_func,
    posterior=result.posterior,
    model_kwargs=result.model_kwargs,
    random_models=100,
    save=False,
    show=True,
)
transient_object.plot_multiband_lightcurve(
    model=model_func,
    posterior=result.posterior,
    model_kwargs=result.model_kwargs,
    random_models=100,
    filters=["ztfg", "ztfr"],
    save=False,
    show=True,
)

# use model based on manually selected parameters and model

In [ ]:
# tde model options:
# Simple analytic fallback
# Cooling envelope (Metzger 2022)
# Gaussian rise + cooling envelope (Sarin and Metzger 2024)
# Broken power law + cooling envelope (Sarin and Metzger 2024)
# Fitted
# TDEMass
# Mosfit TDE

# inspect model priors 
priors = redback.priors.get_priors(model='tde_analytical')                                                                                                                  
print(priors) 

In [ ]:
# # redback has a lot built up to retrieve model parameters with bayesian inference

# # simulate intrinsic TDE lighcurve at z=0
# sampler = 'nestle'
# model = 'tde_analytical'

# priors = redback.priors.get_priors(model=model)                                                                                                                             
# priors['redshift'] = 1e-2                                                                                                                                                   
                                                                                                                                                                            
# model_kwargs = dict(frequency=tde.filtered_frequencies, output_format='flux_density')                                                                                       
# result = redback.fit_model(transient=tde, model=model, sampler=sampler, model_kwargs=model_kwargs,                                                                          
#                             prior=priors, sample='rslice', nlive=200, resume=False)    # make nlive 1000 for actual run                                                                                       
# result.plot_corner()                                                                                                                                                        
# # returns a TDE result object                                                                                                                                               
# result.plot_lightcurve(show=True)                                                                                                                                           
# result.plot_multiband_lightcurve(filters=["g", "r", "i", "z", "y", "J"], show=True)  

In [ ]:
# plot based manually set parameters

from redback.transient_models.tde_models import tde_analytical

z = 2
times = np.linspace(0, 300, 500)
band_freqs = {'u': 8.6e14, 'g': 6.38e14, 'r': 4.858e14, 'i': 4.005e14, 'z': 3.3e14, 'y': 3.0e14}
band_colors = {'u': 'violet', 'g': 'blue', 'r': 'green', 'i': 'orange', 'z': 'red', 'y': 'darkred'}
params = {
    'mej': 1.0, 'vej': 1e4, 'kappa': 0.1, 'kappa_gamma': 10.0,
    'temperature_floor': 5000.0, 'l0': 1e55, 't_0_turn': 50.0
}

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={'width_ratios': [3, 1]})

for band, freq in band_freqs.items():
    flux = tde_analytical(times, redshift=z, **params,
                        frequency=np.full(len(times), freq),
                        output_format='flux_density') * 1e-26  # erg/s/cm²/Hz
    ax.plot(times, flux, color=band_colors[band], label=band)

ax.legend(fontsize=13)
ax.set_xlabel('Time (days)', size=16)
ax.set_ylabel('Flux density (erg/s/cm²/Hz)', size=16)
ax.set_title(f'Example TDE Light Curve z={z} (Analytical Model)', size=18)

ax_text.axis('off')
param_str = f'redshift = {z}\n' + '\n'.join(f'{k} = {v:.3g}' for k, v in params.items())
ax_text.text(0.1, 0.5, param_str, transform=ax_text.transAxes,
            fontsize=12, verticalalignment='center', family='monospace')

plt.tight_layout()
plt.show()

In [ ]:
from redback.transient_models.tde_models import tde_fallback

z=2
times = np.linspace(0, 300, 500)
band_freqs = {'u': 8.6e14, 'g': 6.38e14, 'r': 4.858e14, 'i': 4.005e14, 'z': 3.3e14, 'y': 3.0e14}
band_colors = {'u': 'violet', 'g': 'blue', 'r': 'green', 'i': 'orange', 'z': 'red', 'y': 'darkred'}
params = {
    'mbh6': 1.0,      # BH mass in units of 10^6 solar masses
    'mstar': 1.0,     # disrupted star mass (solar masses)
    'tvisc': 10.0,    # viscous timescale (days)
    'bb': 1.0,        # beta/gamma parameter for disrupted star
    'eta': 0.1,       # BH accretion efficiency
    'leddlimit': 1.0, # Eddington limit factor
    'rph0': 1.0,      # initial photosphere radius
    'lphoto': 1.0,    # photosphere luminosity factor
}

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={'width_ratios': [3, 1]})

for band, freq in band_freqs.items():
    flux = tde_fallback(times, redshift=z, **params,
                        frequency=np.full(len(times), freq),
                        output_format='flux_density') * 1e-26
    ax.plot(times, flux, color=band_colors[band], label=band)

ax.legend(fontsize=13)
ax.set_xlabel('Time (days)', size=16)
ax.set_ylabel('Flux density (erg/s/cm²/Hz)', size=16)
ax.set_title(f'Example TDE Light Curve z={z} (MOSFiT/Fallback Model)', size=18)

ax_text.axis('off')
param_str = f'redshift = {z}\n' + '\n'.join(f'{k} = {v:.3g}' for k, v in params.items())
ax_text.text(0.1, 0.5, param_str, transform=ax_text.transAxes,
            fontsize=12, verticalalignment='center', family='monospace')

plt.tight_layout()
plt.show()

In [ ]:
# compare two redshifts

times = np.linspace(0, 300, 500)
band_names = {'u': 'lsstu', 'g': 'lsstg', 'r': 'lsstr', 'i': 'lssti','z': 'lsstz', 'y': 'lssty'}
band_colors = {'u': 'violet', 'g': 'blue', 'r': 'green', 'i': 'orange', 'z': 'red', 'y': 'darkred'}
params = {                                                                                                                                                        
    'mej': 1.0, 'vej': 1e4, 'kappa': 0.1, 'kappa_gamma': 10.0,                                                                                                  
    'temperature_floor': 5000.0, 'l0': 1e55, 't_0_turn': 50.0                                                                                                     
}                                       
                                                                                                                                                                
fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={'width_ratios': [3, 1]})                                                                    
                                                                                    
for band, band_reg in band_names.items():   
    for z, ls in [(0.01, '-'), (2, '--')]:
        mag = tde_analytical(times, redshift=z, **params,                                                                                                         
                            bands=np.full(len(times), band_reg),                                                                                                 
                            output_format='magnitude')                                                                                                           
        ax.plot(times, mag, color=band_colors[band], linestyle=ls,                                                                                                
                label=f'{band} z={z}') 
                                                                                                                     
                                                                                                                                                                
ax.legend(fontsize=11)                                                                                                                                            
ax.set_xlabel('Time (days)', size=16)                                                                                                                             
ax.invert_yaxis()  
ax.set_ylabel('Apparent magnitude', size=16)   
ax.set_title('Example TDE Light Curve (Analytical Model)', size=18)                                                                                               
                                                                                                                                                                
ax_text.axis('off')
param_str = '\n'.join(f'{k} = {v:.3g}' for k, v in params.items())                                                                                                
ax_text.text(0.1, 0.5, param_str, transform=ax_text.transAxes,
            fontsize=12, verticalalignment='center', family='monospace')                                                                                         
                                                                                                                                                                
plt.tight_layout()                                                                                                                                                
plt.show()

In [ ]:
from redback.transient_models.tde_models import tde_fallback

times = np.linspace(0, 300, 500)
band_names = {'u': 'lsstu', 'g': 'lsstg', 'r': 'lsstr', 'i': 'lssti', 'z': 'lsstz', 'y': 'lssty'}
band_colors = {'u': 'violet', 'g': 'blue', 'r': 'green', 'i': 'orange', 'z': 'red', 'y': 'darkred'}
params = {
    'mbh6': 1.0,
    'mstar': 1.0,
    'tvisc': 10.0,
    'bb': 1.0,
    'eta': 0.1,
    'leddlimit': 1.0,
    'rph0': 1.0,
    'lphoto': 1.0,
}

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={'width_ratios': [3, 1]})

for band, band_reg in band_names.items():
    for z, ls in [(0.01, '-'), (2, '--')]:
        mag = tde_fallback(times, redshift=z, **params,
                            bands=np.full(len(times), band_reg),
                            output_format='magnitude')
        ax.plot(times, mag, color=band_colors[band], linestyle=ls,
                label=f'{band} z={z}')

ax.legend(fontsize=11)
ax.set_xlabel('Time (days)', size=16)
ax.invert_yaxis()
ax.set_ylabel('Apparent magnitude', size=16)
ax.set_title('Example TDE Light Curve (MOSFiT/Fallback Model)', size=18)

ax_text.axis('off')
param_str = '\n'.join(f'{k} = {v:.3g}' for k, v in params.items())
ax_text.text(0.1, 0.5, param_str, transform=ax_text.transAxes,
            fontsize=12, verticalalignment='center', family='monospace')

plt.tight_layout()
plt.show()

In [ ]:
times = np.linspace(0, 300, 500)                                                                                                                                  
band_names = {'u': 'lsstu', 'g': 'lsstg', 'r': 'lsstr', 'i': 'lssti', 'z': 'lsstz', 'y': 'lssty'}                                                                 
band_colors = {'u': 'violet', 'g': 'blue', 'r': 'green', 'i': 'orange', 'z': 'red', 'y': 'darkred'}                                                               
params = {                                                                                                                                                        
    'mej': 1.0, 'vej': 1e4, 'kappa': 0.1, 'kappa_gamma': 10.0,
    'temperature_floor': 5000.0, 'l0': 1e55, 't_0_turn': 50.0                                                                                                     
}                                                                                                                                                                 
redshifts = [(0.1, '-'), (0.5, '--'), (1.0, '-.'), (2.0, ':'), (3.0, (0, (3, 1, 1, 1)))]                                                                          
                                                                                                                                                                
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)                                                                                       
                                                                                                                                                                
for ax, (band, band_reg) in zip(axes.flat, band_names.items()):                                                                                                   
    for z, ls in redshifts:                                                                                                                                       
        mag = tde_analytical(times, redshift=z, **params,                                                                                                         
                            bands=np.full(len(times), band_reg),                                                                                                 
                            output_format='magnitude')                                                                                                         
        abs_mag = mag - cosmo.distmod(z).value                                                                                                                    
        ax.plot(times, abs_mag, color=band_colors[band], linestyle=ls, label=f'z={z}')
    ax.set_title(band, fontsize=14, color=band_colors[band])                                                                                                      
    ax.legend(fontsize=9)                                                                                                                                       
                                        
for ax in axes[1]:                                                                                                                                                
    ax.set_xlabel('Time (days)', size=13)                                                                                                                         
for ax in axes[:, 0]:                                                                                                                                             
    ax.set_ylabel('Absolute magnitude', size=13)                                                                                                                  
                                                                                                                                                                
fig.suptitle('TDE Light Curves by Band (Analytical Model)', size=16)
plt.tight_layout()
ax.invert_yaxis()                                                                                                                                                 
plt.show()

In [ ]:
times = np.linspace(0, 300, 500)            
band_names = {'u': 'lsstu', 'g': 'lsstg', 'r': 'lsstr', 'i': 'lssti', 'z': 'lsstz', 'y': 'lssty'}
band_colors = {'u': 'violet', 'g': 'blue', 'r': 'green', 'i': 'orange', 'z': 'red', 'y': 'darkred'}
params = {                                                                                                                                                        
    'mej': 1.0, 'vej': 1e4, 'kappa': 0.1, 'kappa_gamma': 10.0,
    'temperature_floor': 5000.0, 'l0': 1e55, 't_0_turn': 50.0                                                                                                     
}                                                                                                                                                                 
redshifts = [(0.1, '-'), (0.5, '--'), (1.0, '-.'), (2.0, ':'), (3.0, (0, (3, 1, 1, 1)))]                                                                          
                                                                                                                                                                
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)                                                                                       
                                        
for ax, (band, band_reg) in zip(axes.flat, band_names.items()):                                                                                                   
    for z, ls in redshifts:                                                                                                                                       
        mag = tde_analytical(times, redshift=z, **params,                                                                                                         
                            bands=np.full(len(times), band_reg),                                                                                                 
                            output_format='magnitude')                                                                                                         
        abs_mag = mag - cosmo.distmod(z).value
        ax.plot(times, abs_mag, color=band_colors[band], linestyle=ls)
    ax.set_title(band, fontsize=14, color=band_colors[band])                                                                                                      
                                        
for ax in axes[1]:                                                                                                                                                
    ax.set_xlabel('Time (days)', size=13)                                                                                                                         
for ax in axes[:, 0]:
    ax.set_ylabel('Absolute magnitude', size=13)                                                                                                                  
                                                                                                                                                                
legend_handles = [plt.Line2D([0], [0], color='black', linestyle=ls, label=f'z={z}')
                for z, ls in redshifts]
fig.legend(handles=legend_handles, fontsize=11, loc='upper right',                                                                                                
            bbox_to_anchor=(1.0, 1.0), title='Redshift', title_fontsize=12)                                                                                        
                                                                                                                                                                
fig.suptitle('TDE Light Curves by Band (Analytical Model)', size=16)                                                                                              
plt.tight_layout()                                                                                                                                              
ax.invert_yaxis()                                                                                                                                                 
plt.show() 

In [ ]:
times = np.linspace(0, 1000, 500)
band_names = {'u': 'lsstu', 'g': 'lsstg', 'r': 'lsstr', 'i': 'lssti', 'z': 'lsstz', 'y': 'lssty'}
band_colors = {'u': 'violet', 'g': 'blue', 'r': 'green', 'i': 'orange', 'z': 'red', 'y': 'darkred'}
params = {
    'mbh6': 1.0,
    'mstar': 1.0,
    'tvisc': 10.0,
    'bb': 1.0,
    'eta': 0.1,
    'leddlimit': 1.0,
    'rph0': 1.0,
    'lphoto': 1.0,
}
redshifts = [(0.1, '-'), (0.5, '--'), (1.0, '-.'), (2.0, ':'), (3.0, (0, (3, 1, 1, 1)))]

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)

for ax, (band, band_reg) in zip(axes.flat, band_names.items()):
    for z, ls in redshifts:
        mag = tde_fallback(times, redshift=z, **params,
                            bands=np.full(len(times), band_reg),
                            output_format='magnitude')
        abs_mag = mag - cosmo.distmod(z).value
        ax.plot(times, abs_mag, color=band_colors[band], linestyle=ls)
    ax.set_title(band, fontsize=14, color=band_colors[band])

for ax in axes[1]:
    ax.set_xlabel('Time (days)', size=13)
for ax in axes[:, 0]:
    ax.set_ylabel('Absolute magnitude', size=13)

legend_handles = [plt.Line2D([0], [0], color='black', linestyle=ls, label=f'z={z}')
                for z, ls in redshifts]
fig.legend(handles=legend_handles, fontsize=11, loc='upper right',
            bbox_to_anchor=(1.0, 1.0), title='Redshift', title_fontsize=12)

fig.suptitle('TDE Light Curves by Band (MOSFiT/Fallback Model)', size=16)
plt.tight_layout()
ax.invert_yaxis()
plt.show()

In [ ]:
# FIXME
# Use SimulateGenericTransient only when you want realistic noisy synthetic observations (e.g. for injection-recovery tests). 
# For just visualizing how flux changes with redshift, direct model evaluation is cleaner.

from redback.simulate_transients import SimulateGenericTransient                                                                                                                                                                                                                                                    
                                                                                                                                                                            
# get median posterior parameters from the fit result                                                                                                                       
params_base = {k: v for k, v in result.posterior.median().items() if k not in ['log_likelihood', 'log_prior']}                                                                                                                         
                                                                                                                                                                            
# frequencies for g and i bands (same as used in the fit)                                                                                                                   
frequencies = tde.filtered_frequencies                                                                                                                                      
                                                                                                                                                                            
# observer-frame time grid (days)                                                                                                                                         
times = np.linspace(0, 300, 500)

redshifts = [1.0, 1.5, 2.0, 2.5]                                                                                                                                            

fig, axes = plt.subplots(len(redshifts), 1, figsize=(10, 12), sharex=True)                                                                                                  
                                                                                                                                                                        
for ax, z in zip(axes, redshifts):                                                                                                                                          
    params = params_base.copy()                                                                                                                                           
    params['redshift'] = z                                                                                                                                                  

    sim = SimulateGenericTransient(                                                                                                                                         
        model='tde_analytical',                                                                                                                                           
        parameters=params,
        times=times,
        model_kwargs=dict(frequency=frequencies, output_format='flux_density'),                                                                                             
        data_points=200,
        multiwavelength_transient=True,                                                                                                                                     
        noise_term=0.1,                                                                                                                                                     
    )
                                                                                                                                                                            
    for freq in np.unique(sim.data['frequency']):                                                                                                                           
        mask = sim.data['frequency'] == freq
        ax.errorbar(sim.data['time'][mask], sim.data['output'][mask],                                                                                                       
                    yerr=sim.data['output_error'][mask], fmt='.', label=f'{freq/1e14:.2f}e14 Hz')                                                                           
                                                                                                                                                                            
    ax.set_ylabel('Flux density (mJy)')                                                                                                                                     
    ax.set_title(f'z = {z}')                                                                                                                                                
    ax.legend()                                                                                                                                                             
                                                                                                                                                                        
axes[-1].set_xlabel('Observer-frame time (days)')                                                                                                                           
plt.tight_layout()
plt.show()                            

In [ ]:
# FIXME

#apply extinction

from redback.transient_models.extinction_models import extinction_with_tde_base_model                                                                                       

# best-fit intrinsic parameters from the TDE fit, with extinction added                                                                                                     
params = {k: v for k, v in result.posterior.median().items()                                                                                                              
        if k not in ['log_likelihood', 'log_prior']}                                                                                                                      
params['av_mw'] = 0.1                                                                                                                                                     
params['av_host'] = 0.0                                                                                                                                                     
                                                                                                                                                                        
times = np.linspace(0, 300, 500)  # observer-frame days
frequencies = tde.filtered_frequencies  # g and i band frequencies

fig, ax = plt.subplots(figsize=(10, 5))                                                                                                                                     

for z in [0.01, 1.0, 1.5, 2.0, 2.5]:                                                                                                                                        
    params['redshift'] = z                                                                                                                                                
    flux = extinction_with_tde_base_model(
        times, **params,
        frequency=np.full(len(times), frequencies[0]),                                                                                                                      
        output_format='flux_density',
        base_model='tde_analytical',                                                                                                                                        
    )                                                                                                                                                                     
    ax.plot(times, flux, label=f'z={z}')

ax.set_xlabel('Observer-frame time (days)')
ax.set_ylabel('Flux density (mJy)')
ax.set_title('g-band flux with MW extinction (av_mw=0.1)')
ax.legend()                                                                                                                                                                 
plt.show()

In [ ]:
# FIXME

# add lyman alpha forest absorption to lightcurves

def igm_transmission(freq_obs, z_source):
    """                                                                                                                                                                     
    IGM HI transmission using Madau (1995) Eq. 15.                                                                                                                        
    freq_obs: observed frequency in Hz (scalar or array)                                                                                                                    
    z_source: source redshift                                                                                                                                               
    Returns: transmission array (0 to 1)
    """                                                                                                                                                                     
    c_AA = 2.998e18  # speed of light in Angstrom/s                                                                                                                       
    wave_obs = c_AA / np.atleast_1d(freq_obs)   # observed wavelength in Angstroms                                                                                          
    wave_emit = wave_obs / (1 + z_source)        # rest-frame wavelength                                                                                                    
                                                                                                                                                                            
    T = np.ones_like(wave_obs, dtype=float)                                                                                                                                 
                                                                                                                                                                        
    # complete absorption below Lyman limit (912 Å rest-frame)                                                                                                              
    T[wave_emit < 912.0] = 0.0
                                                                                                                                                                            
    # Lyman-alpha forest: 912 < lambda_rest < 1216 Å                                                                                                                        
    mask = (wave_emit >= 912.0) & (wave_emit < 1216.0)
    if np.any(mask):                                                                                                                                                        
        # effective optical depth: tau = 0.0036 * (lambda_obs / 1216)^3.46                                                                                                
        tau = 0.0036 * (wave_obs[mask] / 1216.0) ** 3.46                                                                                                                    
        T[mask] = np.exp(-tau)
                                                                                                                                                                            
    return T                                                                                                                                                              
                                                                                                                                                                            
                                                                                                                                                                        
# apply to z=2.5 simulation
z = 2.5
params['redshift'] = z
freq = frequencies[0]  # e.g. g-band                                                                                                                                        

flux = extinction_with_tde_base_model(                                                                                                                                      
    times, **params,                                                                                                                                                      
    frequency=np.full(len(times), freq),                                                                                                                                    
    output_format='flux_density',                                                                                                                                         
    base_model='tde_analytical',
)                                                                                                                                                                           

T = igm_transmission(np.array([freq]), z)[0]  # scalar transmission for this band                                                                                           
flux_igm = flux * T                                                                                                                                                       
                                                                                                                                                                            
fig, ax = plt.subplots(figsize=(10, 5))                                                                                                                                     
ax.plot(times, flux, label=f'z={z} no IGM')
ax.plot(times, flux_igm, label=f'z={z} + IGM (Madau 1995)', linestyle='--')                                                                                                 
ax.set_xlabel('Observer-frame time (days)')                                                                                                                                 
ax.set_ylabel('Flux density (mJy)')
ax.set_title(f'g-band transmission at z={z}: T={T:.3f}')                                                                                                                    
ax.legend()                                                                                                                                                                 
plt.show()                     

# Get LSST sampled light curves

### LSST pointing db 

https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/baseline/

In [ ]:
import sqlite3                                                                                                                                                              
import numpy as np                                                                                                                                                        
import pandas as pd                         
import redback                          
from redback.simulate_transients import SimulateOpticalTransient
from redback.transient import OpticalTransient 

In [ ]:
name = 'ZTF18acecugr'
model = 'tde_analytical'

# ── load result ───────────────────────────────────────────────────────────────
result = redback.result.read_in_result(
    outdir=f'redback_inference/{name}/tde_analytical',
    label=f'{name}_nlive500'
)                                                                                                                                                                           
model_func = redback.model_library.all_models_dict[model]
                                                                                                                                                                            
model_kwargs = dict(                                                                                                                                                      
    frequency=transient_object.filtered_frequencies,                                                                                                                        
    output_format='sncosmo_source',                                                                                                                                         
)
result.model_kwargs = model_kwargs                                                                                                                                          
                                                                                                                                                                            
# ── load and reformat OpSim db ───────────────────────────────────────────────                                                                                             
OPSIM_PATH = 'p13_baseline_v5.3.0_10yrs.db'                                                                                                                                 
band_map = {'u': 'lsstu', 'g': 'lsstg', 'r': 'lsstr',                                                                                                                       
            'i': 'lssti', 'z': 'lsstz', 'y': 'lssty'}                                                                                                                       
                                                                                                                                                                            
con = sqlite3.connect(OPSIM_PATH)                                                                                                                                           
pointings = pd.read_sql(                                                                                                                                                    
    "SELECT observationStartMJD, filter, fiveSigmaDepth, fieldRA, fieldDec FROM observations", con)                                                                       
con.close()                                                                                                                                                                 

pointings.rename(columns={'observationStartMJD': 'expMJD'}, inplace=True)                                                                                                   
pointings['filter'] = pointings['filter'].str[0].map(band_map)                                                                                                            
pointings.dropna(subset=['filter'], inplace=True)                                                                                                                           
pointings['_ra']  = np.radians(pointings['fieldRA'])                                                                                                                      
pointings['_dec'] = np.radians(pointings['fieldDec'])                                                                                                                       
                                                                                                                                                                            
# ── simulate with median posterior params ────────────────────────────────────                                                                                             
params = result.posterior.drop(columns=['log_likelihood', 'log_prior'], errors='ignore').median().to_dict()                                                                 
params['t0_mjd_transient'] = 61500.0                                                                                                                                        
                                                                                                                                                                            
sim_name = f'{name}_rubin_sim'                                                                                                                                              
sim = SimulateOpticalTransient.simulate_transient_in_rubin(                                                                                                                 
    model=model,                                                                                                                                                            
    parameters=params,                                                                                                                                                      
    pointings_database=pointings,                                                                                                                                           
    survey=None,                                                                                                                                                            
    model_kwargs={'output_format': 'sncosmo_source'},                                                                                                                       
    snr_threshold=5,
    end_transient_time=500,                                                                                                                                                 
)                                                                                                                                                                         
sim.save_transient(sim_name)                                                                                                                                                
                                                                                                                                                                            
_path = f'simulated/{sim_name}.csv'
_df = pd.read_csv(_path)                                                                                                                                                    
for col in ['magnitude', 'e_magnitude', 'flux_density(mjy)', 'flux_density_error',                                                                                          
            'flux(erg/cm2/s)', 'flux_error', 'limiting_magnitude', 'flux_limit']:                                                                                           
    if col in _df.columns:                                                                                                                                                  
        _df[col] = pd.to_numeric(_df[col], errors='coerce')                                                                                                                 
_df.to_csv(_path, index=False)  

In [ ]:
# plot at different redshifts

for z in [0.05, 0.3, 0.5, 0.7]:               
    params_z = params.copy()                
    params_z['redshift'] = z                                                                                                                                                
    params_z['t0_mjd_transient'] = 61600.0
    params_z['ra']  = np.radians(0.0)                                                                                                                                       
    params_z['dec'] = np.radians(-30.0)                                                                                                                                   
                                                                                                                                                                            
    sim_z = SimulateOpticalTransient.simulate_transient_in_rubin(                                                                                                           
        model='tde_analytical',         
        parameters=params_z,                                                                                                                                                
        pointings_database=pointings,                                                                                                                                       
        survey=None,                                                                                                                                                        
        model_kwargs={'output_format': 'sncosmo_source'},                                                                                                                   
        snr_threshold=5,                                                                                                                                                  
        end_transient_time=300,                                                                                                                                             
    )                                       
    sim_name_z = f'{name}_z{z}_rubin_sim'                                                                                                                                   
    sim_z.save_transient(sim_name_z)                                                                                                                                      
                                                                                                                                                                            
    _path = f'simulated/{sim_name_z}.csv'                                                                                                                                 
    _df = pd.read_csv(_path)                                                                                                                                                
    for col in _df.columns:                                                                                                                                                 
        if col not in ['band', 'system']:                                                                                                                                   
            _df[col] = pd.to_numeric(_df[col], errors='coerce')                                                                                                             
    _df.to_csv(_path, index=False)                                                                                                                                          
                                                                                                                                                                        
    sim_transient_z = OpticalTransient.from_simulated_optical_data(
        name=sim_name_z, data_mode='magnitude', include_upper_limits=True)                                                                                                  

    if len(sim_transient_z.x) == 0:                                                                                                                                         
        print(f"z={z}: no observations in survey window, skipping plots")                                                                                                 
        continue

    posterior_z = result.posterior.copy()                                                                                                                                 
    posterior_z['redshift'] = z                                                                                                                                                               
                                                                                                                                                                            
    sim_transient_z.plot_data(show=True, save=False, title=f'Simulated z={z}')

# lightcurvelynx for simulation

In [ ]:
from lightcurvelynx.models.redback_models import RedbackWrapperModel                                                                                                        
from lightcurvelynx.math_nodes.np_random import NumpyRandomFunc                                                                                                             
from lightcurvelynx.math_nodes.ra_dec_sampler import ObsTableRADECSampler                                                                                                   
from lightcurvelynx.obstable.opsim import OpSim                                                                                                                             
from lightcurvelynx.astro_utils.passbands import PassbandGroup                                                                                                              
from lightcurvelynx.simulate import simulate_lightcurves                                                                                                                    
from lightcurvelynx.survey_info import SurveyInfo                                                                                                                           
from lightcurvelynx.utils.extrapolate import ConstantPadding  

In [ ]:
# load your already-downloaded OpSim DB                                                                                                                                     
opsim_db = OpSim.from_db('p13_baseline_v5.3.0_10yrs.db')                                                                                                                
filters = ['g', 'r', 'i', 'z']                                                                                                                                              
filter_mask = np.isin(opsim_db['filter'], filters)                                                                                                                          
opsim_db = opsim_db.filter_rows(filter_mask)                                                                                                                                
t_min, t_max = opsim_db.time_bounds()                                                                                                                                       
                                                                                                                                                                            
passband_group = PassbandGroup.from_preset(preset='LSST', filters=filters, units='nm')                                                                                      
survey_info = SurveyInfo(obstable=opsim_db, passbands=passband_group)                                                                                                       
                                                                                                                                                                            
# median posterior params as fixed values, redshift as sampler                                                                                                              
params_median = result.posterior.drop(columns=['log_likelihood', 'log_prior'], errors='ignore').median().to_dict()                                                          
                                                                                                                                                                            
parameters = {                                                                                                                                                            
    **params_median,                                                                                                                                                        
    'redshift': NumpyRandomFunc('uniform', low=0.1, high=2.0),  # vary redshift                                                                                           
    # model_kwargs needed by afterglow_and_optical                                                                                                                          
    'base_afterglow_model': 'tophat',                                                                                                                                       
    'base_optical_model': 'evolving_blackbody',                                                                                                                             
    'afterglow_kwargs': {},                                                                                                                                                 
    'optical_kwargs': {},                                                                                                                                                   
    'output_format': 'flux_density',
}                                                                                                                                                                           
                                                                                                                                                                        
ra_dec_sampler = ObsTableRADECSampler(opsim_db, radius=3.0)                                                                                                                 

source = RedbackWrapperModel(                                                                                                                                               
    redback.model_library.all_models_dict['afterglow_and_optical'],                                                                                                       
    parameters=parameters,                                                                                                                                                  
    ra=ra_dec_sampler.ra,                                                                                                                                                   
    dec=ra_dec_sampler.dec,                                                                                                                                                 
    t0=NumpyRandomFunc('uniform', low=t_min, high=t_max),                                                                                                                   
    phase_bounds=(0.1, 300.0),                                                                                                                                            
    time_extrapolation=(ConstantPadding(0.0), ConstantPadding(0.0)),                                                                                                        
)
                                                                                                                                                                            
lightcurves = simulate_lightcurves(source, 100, survey_info)  

# Other objects

In [ ]:
# FIXME:

# repeat for SN1a, PISN, other SLSN